# 📊 RAG Solutions — Metrics & Tradeoff Analysis
Visualizes latency, token usage, and cost across all 7 RAG solutions.

Open 
metrics_analysis.ipynb
 and run all cells. You get 5 graphs saved as PNGs plus a styled summary table:

graph 1 — latency vs output tokens (scatter + trendline)
graph 2 — cost vs total tokens (scatter + trendline)
graph 3 — input vs output token stacked bar
graph 4 — cost per 1000 queries (horizontal bar)
graph 5 — latency vs cost bubble chart (bubble size = total tokens)
summary table with color gradient heatmap
pandas is the only extra needed if not already installed: pip install pandas

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

solutions  = ['E-commerce', 'Healthcare', 'Agriculture', 'Legal', 'Hybrid', 'Plant', 'Study']
latency    = [1.1, 1.3, 1.2, 1.4, 1.6, 2.1, 1.8]
input_tok  = [260, 280, 290, 310, 340, 480, 520]
output_tok = [75,  95,  100, 110, 120, 160, 180]
total_tok  = [i+o for i,o in zip(input_tok, output_tok)]
cost       = [(i*0.59 + o*0.79)/1_000_000 for i,o in zip(input_tok, output_tok)]
colors     = ['#FF6B6B','#4ECDC4','#45B7D1','#96CEB4','#FFEAA7','#DDA0DD','#98D8C8']

print('Data loaded ✅')

: 

## ⏱ Graph 1 — Latency vs Output Tokens

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for s, l, o, c in zip(solutions, latency, output_tok, colors):
    ax.scatter(o, l, color=c, s=250, zorder=5, edgecolors='white', linewidths=1.5)
    ax.annotate(s, (o, l), textcoords='offset points', xytext=(8, 4), fontsize=9)

z = np.polyfit(output_tok, latency, 1)
x_line = np.linspace(min(output_tok)-15, max(output_tok)+15, 100)
ax.plot(x_line, np.poly1d(z)(x_line), 'k--', alpha=0.3, linewidth=1.5, label='Trend')

ax.set_xlabel('Output Tokens', fontsize=11)
ax.set_ylabel('Latency (s)', fontsize=11)
ax.set_title('⏱ Latency vs Output Tokens', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('graph1_latency_vs_tokens.png', dpi=150, bbox_inches='tight')
plt.show()

## 💰 Graph 2 — Cost vs Total Tokens

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for s, t, cv, col in zip(solutions, total_tok, cost, colors):
    ax.scatter(t, cv, color=col, s=250, zorder=5, edgecolors='white', linewidths=1.5)
    ax.annotate(s, (t, cv), textcoords='offset points', xytext=(8, 4), fontsize=9)

z = np.polyfit(total_tok, cost, 1)
x_line = np.linspace(min(total_tok)-20, max(total_tok)+20, 100)
ax.plot(x_line, np.poly1d(z)(x_line), 'k--', alpha=0.3, linewidth=1.5, label='Trend')

ax.set_xlabel('Total Tokens', fontsize=11)
ax.set_ylabel('Cost (USD)', fontsize=11)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'${x:.5f}'))
ax.set_title('💰 Cost vs Total Tokens', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('graph2_cost_vs_tokens.png', dpi=150, bbox_inches='tight')
plt.show()

## 📊 Graph 3 — Input vs Output Token Breakdown

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(solutions))

ax.bar(x, input_tok,  label='Input Tokens',  color='#4ECDC4', alpha=0.85)
ax.bar(x, output_tok, bottom=input_tok, label='Output Tokens', color='#FF6B6B', alpha=0.85)

for i, (inp, out) in enumerate(zip(input_tok, output_tok)):
    ax.text(i, inp/2, str(inp), ha='center', va='center', fontsize=8, color='white', fontweight='bold')
    ax.text(i, inp + out/2, str(out), ha='center', va='center', fontsize=8, color='white', fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(solutions, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Tokens', fontsize=11)
ax.set_title('📊 Input vs Output Token Breakdown', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('graph3_token_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()

## 💸 Graph 4 — Cost per 1000 Queries

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
cost_1k = [c * 1000 for c in cost]

bars = ax.barh(solutions, cost_1k, color=colors, alpha=0.85, edgecolor='white')
ax.bar_label(bars, fmt='$%.3f', padding=5, fontsize=9)

ax.set_xlabel('Cost per 1000 Queries (USD)', fontsize=11)
ax.set_title('💸 Cost per 1000 Queries', fontsize=13, fontweight='bold')
ax.set_xlim(0, max(cost_1k) * 1.3)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('graph4_cost_per_1k.png', dpi=150, bbox_inches='tight')
plt.show()

## 🔄 Graph 5 — Latency vs Cost Tradeoff

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

# bubble size = total tokens
sizes = [t * 0.5 for t in total_tok]
scatter = ax.scatter(latency, cost, s=sizes, c=colors, alpha=0.8, edgecolors='white', linewidths=1.5)

for s, l, cv in zip(solutions, latency, cost):
    ax.annotate(s, (l, cv), textcoords='offset points', xytext=(8, 4), fontsize=9)

ax.set_xlabel('Latency (s)', fontsize=11)
ax.set_ylabel('Cost (USD)', fontsize=11)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'${x:.5f}'))
ax.set_title('🔄 Latency vs Cost (bubble = total tokens)', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('graph5_latency_vs_cost.png', dpi=150, bbox_inches='tight')
plt.show()

## 📋 Summary Table

In [ ]:
import pandas as pd

df = pd.DataFrame({
    'Solution':       solutions,
    'Latency (s)':    latency,
    'Input Tokens':   input_tok,
    'Output Tokens':  output_tok,
    'Total Tokens':   total_tok,
    'Cost/Query ($)': [f'${c:.6f}' for c in cost],
    'Cost/1K ($)':    [f'${c*1000:.3f}' for c in cost],
})

df.set_index('Solution', inplace=True)
df.style.background_gradient(subset=['Latency (s)', 'Total Tokens'], cmap='YlOrRd')